In [1]:
import torch

In [2]:
class HebbNeuron:
    
    def __init__(self, input_size):
        # Inicializar los pesos en cero
        self.w = torch.zeros(input_size, requires_grad=False)
        
        # Inicializar el bias en cero
        self.b = 0.0

    def forward(self, x):
        """
        Calcula la salida de la neurona.
        
        Si la entrada neta es mayor o igual a cero,
        la salida es 1. De lo contrario, es -1.
        """
        
        y_in = torch.sum(x * self.w) + self.b
        
        return 1 if y_in >= 0 else -1

    def train(self, S, T):
        """
        Entrena la neurona utilizando la regla de Hebb.
        
        S: conjunto de entradas.
        T: salidas esperadas.
        """
        
        for i in range(len(S)):
            x = S[i, :]
            y = T[i]

            # Actualización de pesos
            self.w = self.w + x * y

            # Actualización del bias
            self.b = self.b + y

In [3]:

class PerceptronNeuron:
    
    def __init__(self, input_size):
        # Se agrega una posición adicional para el bias
        self.w = torch.zeros(
            input_size + 1,
            1,
            dtype=torch.float32
        )

        self.threshold = 0
        self.learning_rate = 0

    def forward(self, x):
        """
        Calcula la salida del perceptrón.
        
        w[0] representa el bias.
        w[1] representa el peso de x1.
        w[2] representa el peso de x2.
        """

        # Agregar el valor 1 correspondiente al bias
        x = torch.cat(
            (torch.tensor([1]), x)
        ).to(dtype=torch.float32)

        # Calcular la entrada neta
        y_in = torch.matmul(x, self.w)

        # Función de activación con umbral
        if y_in > self.threshold:
            y = 1

        elif y_in < -self.threshold:
            y = -1

        else:
            y = 0

        return y

    def train(self, S, T, threshold, learning_rate):
        """
        Entrena el perceptrón hasta clasificar correctamente
        todas las entradas.
        """

        self.threshold = threshold
        self.learning_rate = learning_rate

        stop_condition = False
        epoch = 0

        while not stop_condition:

            # Se supone inicialmente que todos los patrones
            # fueron clasificados correctamente
            stop_condition = True

            for i in range(len(S)):

                x = S[i, :]
                t = T[i]

                # Obtener predicción
                y = self.forward(x)

                # Actualizar solamente si existe un error
                if y != t:

                    x_column = x.to(
                        dtype=torch.float32
                    ).reshape(2, 1)

                    # Actualización de los pesos
                    self.w[1:] = (
                        self.w[1:]
                        + self.learning_rate * x_column * t
                    )

                    # Actualización del bias
                    self.w[0] = (
                        self.w[0]
                        + self.learning_rate * t
                    )

                    stop_condition = False

            epoch += 1

            print(f"\nÉpoca {epoch}")
            print(self.w)

In [4]:
# ============================================================
# DATOS DE ENTRADA
# ============================================================

S = torch.tensor([
    [ 1,  1],     # Equivale a 1, 1
    [ 1, -1],     # Equivale a 1, 0
    [-1,  1],     # Equivale a 0, 1
    [-1, -1]      # Equivale a 0, 0
])

print("Entradas bipolares:")
print(S)

Entradas bipolares:
tensor([[ 1,  1],
        [ 1, -1],
        [-1,  1],
        [-1, -1]])


In [5]:
# ============================================================
# SALIDAS ESPERADAS
# ============================================================

# Tabla de verdad bipolar para OR
T_OR = torch.tensor([
     1,
     1,
     1,
    -1
])

# Tabla de verdad bipolar para x1 AND NOT x2
T_AND_NOT = torch.tensor([
    -1,
     1,
    -1,
    -1
])

print("Salidas de OR:")
print(T_OR)

print("\nSalidas de AND NOT:")
print(T_AND_NOT)

Salidas de OR:
tensor([ 1,  1,  1, -1])

Salidas de AND NOT:
tensor([-1,  1, -1, -1])


In [6]:
# ============================================================
# FUNCIÓN PARA MOSTRAR RESULTADOS
# ============================================================

def convertir_a_binario(valor):
    """
    Convierte valores bipolares a binarios:
    
     1 -> 1
    -1 -> 0
    """
    
    return 1 if int(valor) == 1 else 0


def mostrar_resultados(nombre, neurona, entradas, targets):
    
    print("\n" + "=" * 55)
    print(f"RESULTADOS DE LA COMPUERTA {nombre}")
    print("=" * 55)

    print("x1\tx2\tEsperado\tObtenido")
    print("-" * 40)

    correctos = 0

    for x, target in zip(entradas, targets):

        resultado = neurona.forward(x)

        # Convertir entradas de bipolar a binario
        x1_binario = convertir_a_binario(x[0])
        x2_binario = convertir_a_binario(x[1])

        # Convertir resultados de bipolar a binario
        esperado_binario = convertir_a_binario(target)
        obtenido_binario = convertir_a_binario(resultado)

        if esperado_binario == obtenido_binario:
            correctos += 1

        print(
            f"{x1_binario}\t"
            f"{x2_binario}\t"
            f"{esperado_binario}\t\t"
            f"{obtenido_binario}"
        )

    porcentaje = correctos / len(entradas) * 100

    print("-" * 40)
    print(f"Aciertos: {correctos} de {len(entradas)}")
    print(f"Exactitud: {porcentaje:.2f}%")

In [7]:
# ============================================================
# ENTRENAR OR CON HEBB
# ============================================================

hebb_or = HebbNeuron(input_size=2)

hebb_or.train(S, T_OR)

print("Pesos obtenidos para OR con Hebb:")
print("w1 =", hebb_or.w[0].item())
print("w2 =", hebb_or.w[1].item())
print("Bias =", hebb_or.b.item())

mostrar_resultados(
    nombre="OR CON HEBB",
    neurona=hebb_or,
    entradas=S,
    targets=T_OR
)

Pesos obtenidos para OR con Hebb:
w1 = 2.0
w2 = 2.0
Bias = 2.0

RESULTADOS DE LA COMPUERTA OR CON HEBB
x1	x2	Esperado	Obtenido
----------------------------------------
1	1	1		1
1	0	1		1
0	1	1		1
0	0	0		0
----------------------------------------
Aciertos: 4 de 4
Exactitud: 100.00%


In [8]:
# ============================================================
# ENTRENAR AND NOT CON HEBB
# ============================================================

hebb_and_not = HebbNeuron(input_size=2)

hebb_and_not.train(S, T_AND_NOT)

print("Pesos obtenidos para AND NOT con Hebb:")
print("w1 =", hebb_and_not.w[0].item())
print("w2 =", hebb_and_not.w[1].item())
print("Bias =", hebb_and_not.b.item())

mostrar_resultados(
    nombre="AND NOT CON HEBB",
    neurona=hebb_and_not,
    entradas=S,
    targets=T_AND_NOT
)

Pesos obtenidos para AND NOT con Hebb:
w1 = 2.0
w2 = -2.0
Bias = -2.0

RESULTADOS DE LA COMPUERTA AND NOT CON HEBB
x1	x2	Esperado	Obtenido
----------------------------------------
1	1	0		0
1	0	1		1
0	1	0		0
0	0	0		0
----------------------------------------
Aciertos: 4 de 4
Exactitud: 100.00%


In [9]:
# ============================================================
# ENTRENAR OR CON PERCEPTRÓN
# ============================================================

perceptron_or = PerceptronNeuron(input_size=2)

perceptron_or.train(
    S=S,
    T=T_OR,
    threshold=1,
    learning_rate=1
)

print("\nPesos finales para OR con perceptrón:")
print("Bias =", perceptron_or.w[0].item())
print("w1 =", perceptron_or.w[1].item())
print("w2 =", perceptron_or.w[2].item())
print("Umbral =", perceptron_or.threshold)

mostrar_resultados(
    nombre="OR CON PERCEPTRÓN",
    neurona=perceptron_or,
    entradas=S,
    targets=T_OR
)


Época 1
tensor([[2.],
        [2.],
        [2.]])

Época 2
tensor([[2.],
        [2.],
        [2.]])

Pesos finales para OR con perceptrón:
Bias = 2.0
w1 = 2.0
w2 = 2.0
Umbral = 1

RESULTADOS DE LA COMPUERTA OR CON PERCEPTRÓN
x1	x2	Esperado	Obtenido
----------------------------------------
1	1	1		1
1	0	1		1
0	1	1		1
0	0	0		0
----------------------------------------
Aciertos: 4 de 4
Exactitud: 100.00%


In [10]:
# ============================================================
# ENTRENAR AND NOT CON PERCEPTRÓN
# ============================================================

perceptron_and_not = PerceptronNeuron(input_size=2)

perceptron_and_not.train(
    S=S,
    T=T_AND_NOT,
    threshold=1,
    learning_rate=1
)

print("\nPesos finales para AND NOT con perceptrón:")
print("Bias =", perceptron_and_not.w[0].item())
print("w1 =", perceptron_and_not.w[1].item())
print("w2 =", perceptron_and_not.w[2].item())
print("Umbral =", perceptron_and_not.threshold)

mostrar_resultados(
    nombre="AND NOT CON PERCEPTRÓN",
    neurona=perceptron_and_not,
    entradas=S,
    targets=T_AND_NOT
)


Época 1
tensor([[-1.],
        [ 1.],
        [-1.]])

Época 2
tensor([[-2.],
        [ 2.],
        [-2.]])

Época 3
tensor([[-2.],
        [ 2.],
        [-2.]])

Pesos finales para AND NOT con perceptrón:
Bias = -2.0
w1 = 2.0
w2 = -2.0
Umbral = 1

RESULTADOS DE LA COMPUERTA AND NOT CON PERCEPTRÓN
x1	x2	Esperado	Obtenido
----------------------------------------
1	1	0		0
1	0	1		1
0	1	0		0
0	0	0		0
----------------------------------------
Aciertos: 4 de 4
Exactitud: 100.00%


In [11]:
# ============================================================
# RESUMEN FINAL
# ============================================================

print("\n" + "=" * 65)
print("RESUMEN DE PESOS Y UMBRALES")
print("=" * 65)

print("\nOR con Hebb")
print("w1:", hebb_or.w[0].item())
print("w2:", hebb_or.w[1].item())
print("Bias:", hebb_or.b.item())
print("Umbral implícito: 0")

print("\nAND NOT con Hebb")
print("w1:", hebb_and_not.w[0].item())
print("w2:", hebb_and_not.w[1].item())
print("Bias:", hebb_and_not.b.item())
print("Umbral implícito: 0")

print("\nOR con Perceptrón")
print("w1:", perceptron_or.w[1].item())
print("w2:", perceptron_or.w[2].item())
print("Bias:", perceptron_or.w[0].item())
print("Umbral:", perceptron_or.threshold)

print("\nAND NOT con Perceptrón")
print("w1:", perceptron_and_not.w[1].item())
print("w2:", perceptron_and_not.w[2].item())
print("Bias:", perceptron_and_not.w[0].item())
print("Umbral:", perceptron_and_not.threshold)


RESUMEN DE PESOS Y UMBRALES

OR con Hebb
w1: 2.0
w2: 2.0
Bias: 2.0
Umbral implícito: 0

AND NOT con Hebb
w1: 2.0
w2: -2.0
Bias: -2.0
Umbral implícito: 0

OR con Perceptrón
w1: 2.0
w2: 2.0
Bias: 2.0
Umbral: 1

AND NOT con Perceptrón
w1: 2.0
w2: -2.0
Bias: -2.0
Umbral: 1
